In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("FlowCast - LSTM Traffic Forecasting")
print("Libraries imported successfully.")

FlowCast - LSTM Traffic Forecasting
Libraries imported successfully.


In [5]:
data_path = "../data/processed/flowcast_merged.csv"

if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Dataset not found at: {os.path.abspath(data_path)}"
    )

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

Dataset loaded successfully.
Dataset shape: (176701, 31)

Column names:
['road_id', 'road_name', 'latitude', 'longitude', 'weather_station_id', 'date', 'time', 'traffic_volume', 'vehicle_count', 'vehicle_type_dist', 'avg_speed', 'occupancy', 'congestion_level', 'travel_time', 'accident_count', 'signal_timing', 'road_capacity', 'timestamp', 'capacity_30min', 'volume_capacity_ratio', 'date_weather', 'time_weather', 'weather_condition', 'temperature', 'rainfall', 'visibility', 'public_holiday', 'holiday_name', 'event_flag', 'event_name', 'roadwork_flag']


C:\Users\pathi\AppData\Local\Temp\ipykernel_16440\813702849.py:8: DtypeWarning: Columns (27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


,road_id,road_name,latitude,longitude,weather_station_id,date,time,traffic_volume,vehicle_count,vehicle_type_dist,...,time_weather,weather_condition,temperature,rainfall,visibility,public_holiday,holiday_name,event_flag,event_name,roadwork_flag
0,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-01,00:00,97.0,97,"{""2W"": 0.39, ""Car"": 0.35, ""LCV"": 0.12, ""HCV"": ...",...,00:00,Clear,12.7,0.0,10000.0,1,New Year's Day,0,NaN,0
1,NL-012,Market St @ Exchange,28.671280,77.258765,WS-SOUTH,2025-01-01,00:00,72.0,72,"{""2W"": 0.43, ""Car"": 0.36, ""LCV"": 0.12, ""HCV"": ...",...,00:00,Clear,7.6,0.0,10000.0,1,New Year's Day,0,NaN,0
2,NL-015,Airport Feeder,28.626508,77.212314,WS-SOUTH,2025-01-01,00:00,115.0,115,"{""2W"": 0.4, ""Car"": 0.35, ""LCV"": 0.16, ""HCV"": 0...",...,00:00,Clear,7.6,0.0,10000.0,1,New Year's Day,0,NaN,0
3,NL-013,Industrial Link A,28.647981,77.161851,WS-NORTH,2025-01-01,00:00,73.0,73,"{""2W"": 0.4, ""Car"": 0.4, ""LCV"": 0.1, ""HCV"": 0.1}",...,00:00,Clear,12.7,0.0,10000.0,1,New Year's Day,0,NaN,0
4,NL-014,Industrial Link B,28.594749,77.253501,WS-CENTRAL,2025-01-01,00:00,119.0,119,"{""2W"": 0.4, ""Car"": 0.33, ""LCV"": 0.16, ""HCV"": 0...",...,00:00,Clear,10.2,0.0,10000.0,1,New Year's Day,0,NaN,0


In [6]:
required_columns = ["road_id", "date", "time", "traffic_volume"]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("Columns needing attention:", missing_columns)
else:
    print("Required columns are available.")

print("\nMissing values in key columns:")
display(df[required_columns].isnull().sum())

Required columns are available.

Missing values in key columns:


road_id           0
date              0
time              0
traffic_volume    0
dtype: int64

In [7]:
# Combine date and time into a single timestamp
df["timestamp"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["time"].astype(str),
    errors="coerce"
)

# Check timestamp conversion
print("Timestamp preparation completed.")
print("Invalid timestamps:", df["timestamp"].isna().sum())

display(df[["road_id", "date", "time", "timestamp", "traffic_volume"]].head())

Timestamp preparation completed.
Invalid timestamps: 0


,road_id,date,time,timestamp,traffic_volume
0,NL-001,2025-01-01,00:00,2025-01-01,97.0
1,NL-012,2025-01-01,00:00,2025-01-01,72.0
2,NL-015,2025-01-01,00:00,2025-01-01,115.0
3,NL-013,2025-01-01,00:00,2025-01-01,73.0
4,NL-014,2025-01-01,00:00,2025-01-01,119.0


In [8]:
df = df.dropna(subset=["timestamp", "traffic_volume"]).copy()

df = df.sort_values(
    ["road_id", "timestamp"]
).reset_index(drop=True)

print("Data sorted by road and time.")
print("Total rows:", len(df))
print("Number of road segments:", df["road_id"].nunique())

display(df[["road_id", "timestamp", "traffic_volume"]].head(10))

Data sorted by road and time.
Total rows: 176701
Number of road segments: 25


,road_id,timestamp,traffic_volume
0,NL-001,2025-01-01 00:00:00,97.0
1,NL-001,2025-01-01 00:30:00,99.0
2,NL-001,2025-01-01 01:00:00,102.0
3,NL-001,2025-01-01 01:30:00,113.0
4,NL-001,2025-01-01 02:00:00,92.0
5,NL-001,2025-01-01 02:30:00,103.0
6,NL-001,2025-01-01 03:00:00,94.0
7,NL-001,2025-01-01 03:30:00,91.0
8,NL-001,2025-01-01 04:30:00,104.0
9,NL-001,2025-01-01 05:00:00,108.0


In [9]:
print("Timestamp data type:", df["timestamp"].dtype)

display(
    df[["road_id", "date", "time", "timestamp", "traffic_volume"]]
    .head(10)
    .to_string(index=False)
)

print("\nUnique time intervals:")
print(
    df.groupby("road_id")["timestamp"]
      .apply(lambda s: s.sort_values().diff().dropna().value_counts().head(3))
      .head(10)
)

Timestamp data type: datetime64[ns]


'road_id       date  time           timestamp  traffic_volume\n NL-001 2025-01-01 00:00 2025-01-01 00:00:00            97.0\n NL-001 2025-01-01 00:30 2025-01-01 00:30:00            99.0\n NL-001 2025-01-01 01:00 2025-01-01 01:00:00           102.0\n NL-001 2025-01-01 01:30 2025-01-01 01:30:00           113.0\n NL-001 2025-01-01 02:00 2025-01-01 02:00:00            92.0\n NL-001 2025-01-01 02:30 2025-01-01 02:30:00           103.0\n NL-001 2025-01-01 03:00 2025-01-01 03:00:00            94.0\n NL-001 2025-01-01 03:30 2025-01-01 03:30:00            91.0\n NL-001 2025-01-01 04:30 2025-01-01 04:30:00           104.0\n NL-001 2025-01-01 05:00 2025-01-01 05:00:00           108.0'


Unique time intervals:
road_id  NL-001         
NL-001   0 days 00:30:00    6874
         0 days 01:00:00     179
         0 days 01:30:00       5
NL-002   0 days 00:30:00    6917
         0 days 01:00:00     159
         0 days 01:30:00       4
NL-003   0 days 00:30:00    6881
         0 days 01:00:00     180
         0 days 01:30:00       2
NL-004   0 days 00:30:00    6936
Name: timestamp, dtype: int64


In [10]:
# Use a shared time cutoff so all roads follow the same timeline
cutoff = df["timestamp"].quantile(0.80)

train_df = df[df["timestamp"] < cutoff].copy()
test_df = df[df["timestamp"] >= cutoff].copy()

print("Time cutoff:", cutoff)
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining period:")
print(train_df["timestamp"].min(), "to", train_df["timestamp"].max())

print("\nTesting period:")
print(test_df["timestamp"].min(), "to", test_df["timestamp"].max())

Time cutoff: 2025-05-01 18:30:00
Training rows: 141342
Testing rows: 35359

Training period:
2025-01-01 00:00:00 to 2025-05-01 18:00:00

Testing period:
2025-05-01 18:30:00 to 2025-05-31 23:30:00


In [11]:
import tensorflow

print("Imported module:", tensorflow)
print("Module type:", type(tensorflow))
print("Module location:", getattr(tensorflow, "_path_", "Not available"))
print("Available attributes:", dir(tensorflow)[:30])

Imported module: <module 'tensorflow' from 'C:\\Users\\pathi\\anaconda3\\Lib\\site-packages\\tensorflow\\__init__.py'>
Module type: <class 'module'>
Module location: Not available
Available attributes: ['AggregationMethod', 'Assert', 'CriticalSection', 'DType', 'DeviceSpec', 'GradientTape', 'Graph', 'IndexedSlices', 'IndexedSlicesSpec', 'Module', 'Operation', 'OptionalSpec', 'RaggedTensor', 'RaggedTensorSpec', 'RegisterGradient', 'SparseTensor', 'SparseTensorSpec', 'Tensor', 'TensorArray', 'TensorArraySpec', 'TensorShape', 'TensorSpec', 'TypeSpec', 'UnconnectedGradients', 'Variable', 'VariableAggregation', 'VariableSynchronization', '_API_MODULE', '_KerasLazyLoader', '__all__']


In [12]:
import importlib.metadata as metadata

try:
    print("TensorFlow package version:", metadata.version("tensorflow"))
except metadata.PackageNotFoundError:
    print("TensorFlow package is not registered as installed.")

try:
    print("Keras package version:", metadata.version("keras"))
except metadata.PackageNotFoundError:
    print("Keras package is not registered as installed.")

TensorFlow package version: 2.20.0
Keras package version: 3.15.1


In [13]:
import tensorflow as tf
import tensorflow as tf
import keras
from importlib.metadata import version

print("TensorFlow version:", tf.version.VERSION)
print("Keras package version:", version("keras"))

# Test LSTM model creation
model = keras.Sequential([
    keras.layers.Input(shape=(4, 3)),
    keras.layers.LSTM(16),
    keras.layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse")

print("LSTM model created successfully!")
model.summary()

TensorFlow version: 2.20.0
Keras package version: 3.15.1
LSTM model created successfully!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 16)                  │           1,280 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,297 (5.07 KB)

 Trainable params: 1,297 (5.07 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# Check the dataset before preparing LSTM sequences

print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

print("\nRequired columns:")
for col in ["road_id", "timestamp", "traffic_volume"]:
    print(f"{col}: {'Available' if col in df.columns else 'MISSING'}")

print("\nTimestamp data type:", df["timestamp"].dtype)
print("\nMissing traffic_volume values:", df["traffic_volume"].isna().sum())
print("\nMissing timestamp values:", df["timestamp"].isna().sum())

Dataset shape: (176701, 31)

Available columns:
['road_id', 'road_name', 'latitude', 'longitude', 'weather_station_id', 'date', 'time', 'traffic_volume', 'vehicle_count', 'vehicle_type_dist', 'avg_speed', 'occupancy', 'congestion_level', 'travel_time', 'accident_count', 'signal_timing', 'road_capacity', 'timestamp', 'capacity_30min', 'volume_capacity_ratio', 'date_weather', 'time_weather', 'weather_condition', 'temperature', 'rainfall', 'visibility', 'public_holiday', 'holiday_name', 'event_flag', 'event_name', 'roadwork_flag']

Required columns:
road_id: Available
timestamp: Available
traffic_volume: Available

Timestamp data type: datetime64[ns]

Missing traffic_volume values: 0

Missing timestamp values: 0


In [15]:
import numpy as np
import pandas as pd

# Work on a separate copy so the original dataset remains unchanged
lstm_df = df[["road_id", "timestamp", "traffic_volume"]].copy()

lstm_df = lstm_df.sort_values(
    ["road_id", "timestamp"]
).reset_index(drop=True)

# Check for duplicate road/time records
duplicate_count = lstm_df.duplicated(
    subset=["road_id", "timestamp"]
).sum()

print("Duplicate road/time records:", duplicate_count)

# Check the time intervals for the first road
first_road = lstm_df["road_id"].iloc[0]
road_data = lstm_df[lstm_df["road_id"] == first_road].copy()

print("\nExample road:", first_road)
print("Rows:", len(road_data))
print("Time range:", road_data["timestamp"].min(),
      "to", road_data["timestamp"].max())

print("\nTraffic volume summary:")
print(lstm_df["traffic_volume"].describe())

Duplicate road/time records: 0

Example road: NL-001
Rows: 7059
Time range: 2025-01-01 00:00:00 to 2025-05-31 23:30:00

Traffic volume summary:
count    176701.000000
mean        431.436582
std         283.208213
min          41.000000
25%         195.000000
50%         362.000000
75%         601.000000
max        2090.000000
Name: traffic_volume, dtype: float64


In [16]:
# Check whether each road has regular 30-minute intervals

lstm_df = lstm_df.sort_values(
    ["road_id", "timestamp"]
).reset_index(drop=True)

lstm_df["time_difference"] = (
    lstm_df.groupby("road_id")["timestamp"].diff()
)

print("Time interval counts:")
print(lstm_df["time_difference"].value_counts().head(10))

print("\nIntervals other than 30 minutes:")
print(
    lstm_df.loc[
        lstm_df["time_difference"].notna()
        & (lstm_df["time_difference"] != pd.Timedelta(minutes=30)),
        ["road_id", "timestamp", "time_difference"]
    ].head(10)
)

print("\nTotal irregular intervals:",
      (
          lstm_df["time_difference"].notna()
          & (lstm_df["time_difference"] != pd.Timedelta(minutes=30))
      ).sum())

Time interval counts:
time_difference
0 days 00:30:00    172291
0 days 01:00:00      4272
0 days 01:30:00       113
Name: count, dtype: int64

Intervals other than 30 minutes:
    road_id           timestamp time_difference
8    NL-001 2025-01-01 04:30:00 0 days 01:00:00
51   NL-001 2025-01-02 02:30:00 0 days 01:00:00
53   NL-001 2025-01-02 04:00:00 0 days 01:00:00
111  NL-001 2025-01-03 09:30:00 0 days 01:00:00
143  NL-001 2025-01-04 02:00:00 0 days 01:00:00
187  NL-001 2025-01-05 00:30:00 0 days 01:00:00
219  NL-001 2025-01-05 17:00:00 0 days 01:00:00
227  NL-001 2025-01-05 22:00:00 0 days 01:30:00
233  NL-001 2025-01-06 01:30:00 0 days 01:00:00
250  NL-001 2025-01-06 10:30:00 0 days 01:00:00

Total irregular intervals: 4385


C:\Users\pathi\AppData\Local\Temp\ipykernel_16440\2095198434.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  & (lstm_df["time_difference"] != pd.Timedelta(minutes=30)),
C:\Users\pathi\AppData\Local\Temp\ipykernel_16440\2095198434.py:26: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  & (lstm_df["time_difference"] != pd.Timedelta(minutes=30))


In [17]:
# Split each road into continuous 30-minute segments

lstm_df = lstm_df.sort_values(
    ["road_id", "timestamp"]
).reset_index(drop=True)

lstm_df["time_difference"] = (
    lstm_df.groupby("road_id")["timestamp"].diff()
)

# Start a new segment whenever the interval is not 30 minutes
lstm_df["segment_id"] = (
    lstm_df["time_difference"] != pd.Timedelta(minutes=30)
).groupby(lstm_df["road_id"]).cumsum()

# Count the continuous segments
segment_summary = (
    lstm_df.groupby(["road_id", "segment_id"])
    .size()
    .reset_index(name="rows")
)

print("Total continuous segments:", len(segment_summary))
print("\nSegment length summary:")
print(segment_summary["rows"].describe())

print("\nSegments with at least 5 readings:",
      (segment_summary["rows"] >= 5).sum())

print("\nSegments with at least 10 readings:",
      (segment_summary["rows"] >= 10).sum())

Total continuous segments: 4410

Segment length summary:
count    4410.000000
mean       40.068254
std        40.157510
min         1.000000
25%        12.000000
50%        27.000000
75%        54.000000
max       338.000000
Name: rows, dtype: float64

Segments with at least 5 readings: 3988

Segments with at least 10 readings: 3530


C:\Users\pathi\AppData\Local\Temp\ipykernel_16440\411569916.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  lstm_df["time_difference"] != pd.Timedelta(minutes=30)


In [18]:
# Create sequences for LSTM traffic-volume forecasting

import numpy as np

SEQUENCE_LENGTH = 12  # Previous 6 hours (12 readings × 30 minutes)
FORECAST_STEPS = 1    # Predict the next 30-minute traffic volume

X_sequences = []
y_sequences = []
sequence_road_ids = []
sequence_timestamps = []

for (road_id, segment_id), group in lstm_df.groupby(
    ["road_id", "segment_id"], sort=False
):
    group = group.sort_values("timestamp")

    values = group["traffic_volume"].to_numpy(dtype=np.float32)
    timestamps = group["timestamp"].to_numpy()

    # Need 12 previous readings plus 1 future target
    if len(values) < SEQUENCE_LENGTH + FORECAST_STEPS:
        continue

    for i in range(len(values) - SEQUENCE_LENGTH):
        X_sequences.append(values[i:i + SEQUENCE_LENGTH])
        y_sequences.append(values[i + SEQUENCE_LENGTH])
        sequence_road_ids.append(road_id)
        sequence_timestamps.append(timestamps[i + SEQUENCE_LENGTH])

X_sequences = np.array(X_sequences, dtype=np.float32)
y_sequences = np.array(y_sequences, dtype=np.float32)
sequence_road_ids = np.array(sequence_road_ids)
sequence_timestamps = pd.to_datetime(sequence_timestamps)

# LSTM expects 3 dimensions: samples, time steps, features
X_sequences = X_sequences.reshape(
    X_sequences.shape[0], SEQUENCE_LENGTH, 1
)

print("X shape:", X_sequences.shape)
print("y shape:", y_sequences.shape)
print("Road IDs:", len(np.unique(sequence_road_ids)))
print("Target timestamps:", len(sequence_timestamps))
print("All X values finite:", np.isfinite(X_sequences).all())
print("All y values finite:", np.isfinite(y_sequences).all())

X shape: (130332, 12, 1)
y shape: (130332,)
Road IDs: 25
Target timestamps: 130332
All X values finite: True
All y values finite: True


In [19]:
# Correct chronological split across all roads

sequence_order = np.argsort(sequence_timestamps)

X_ordered = X_sequences[sequence_order]
y_ordered = y_sequences[sequence_order]
timestamps_ordered = sequence_timestamps[sequence_order]

unique_times = np.sort(np.unique(timestamps_ordered))
split_index = int(len(unique_times) * 0.8)
cutoff_time = unique_times[split_index]

train_mask = timestamps_ordered < cutoff_time
test_mask = timestamps_ordered >= cutoff_time

X_train = X_ordered[train_mask]
y_train = y_ordered[train_mask]
X_test = X_ordered[test_mask]
y_test = y_ordered[test_mask]

train_timestamps = timestamps_ordered[train_mask]
test_timestamps = timestamps_ordered[test_mask]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nTraining period:", train_timestamps.min(), "to", train_timestamps.max())
print("Testing period:", test_timestamps.min(), "to", test_timestamps.max())
print("\nX_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("\nNo timestamp overlap:",
      train_timestamps.max() < test_timestamps.min())

Training samples: 104502
Testing samples: 25830

Training period: 2025-01-01 06:00:00 to 2025-05-01 19:30:00
Testing period: 2025-05-01 20:00:00 to 2025-05-31 23:30:00

X_train: (104502, 12, 1)
X_test: (25830, 12, 1)
y_train: (104502,)
y_test: (25830,)

No timestamp overlap: True


In [20]:
from sklearn.preprocessing import MinMaxScaler

# Fit the scaler using training targets only
scaler = MinMaxScaler()

y_train_scaled = scaler.fit_transform(
    y_train.reshape(-1, 1)
).astype(np.float32).ravel()

y_test_scaled = scaler.transform(
    y_test.reshape(-1, 1)
).astype(np.float32).ravel()

# Scale each input sequence using the same training-fitted scaler
X_train_scaled = scaler.transform(
    X_train.reshape(-1, 1)
).reshape(X_train.shape).astype(np.float32)

X_test_scaled = scaler.transform(
    X_test.reshape(-1, 1)
).reshape(X_test.shape).astype(np.float32)

print("Scaling completed successfully.")
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)
print("y_train_scaled:", y_train_scaled.shape)
print("y_test_scaled:", y_test_scaled.shape)
print("Training values finite:", np.isfinite(X_train_scaled).all())
print("Testing values finite:", np.isfinite(X_test_scaled).all())

Scaling completed successfully.
X_train_scaled: (104502, 12, 1)
X_test_scaled: (25830, 12, 1)
y_train_scaled: (104502,)
y_test_scaled: (25830,)
Training values finite: True
Testing values finite: True


In [21]:
# Build and train the LSTM traffic-volume forecasting model

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1], 1)),
    layers.LSTM(32),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_split=0.1,
    epochs=15,
    batch_size=256,
    callbacks=[early_stopping],
    shuffle=False,
    verbose=1
)

print("LSTM training completed successfully.")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                        │ (None, 32)                  │           4,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,897 (19.13 KB)

 Trainable params: 4,897 (19.13 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
368/368 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 0.0085 - mae: 0.0679 - val_loss: 0.0030 - val_mae: 0.0401
Epoch 2/15
368/368 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.0022 - mae: 0.0340 - val_loss: 0.0019 - val_mae: 0.0316
Epoch 3/15
368/368 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.0019 - mae: 0.0306 - val_loss: 0.0018 - val_mae: 0.0306
Epoch 4/15
368/368 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.0018 - mae: 0.0299 - val_loss: 0.0017 - val_mae: 0.0302
Epoch 5/15
368/368 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0017 - mae: 0.0292 - val_loss: 0.0017 - val_mae: 0.0300
Epoch 6/15
368/368 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.0016 - mae: 0.0287 - val_loss: 0.0016 - val_mae: 0.0292
Epoch 7/15
368/368 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 0.0016 - mae: 0.0280 - val_loss: 0.0015 - val_mae: 0.0281
Epoch 8/15
368/368 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0015 - mae: 0.0272 - val_loss: 0.0014 - val_mae: 0.0268
Epoch 9/15
368/368 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/

In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Generate predictions on the test data
y_pred_scaled = model.predict(X_test_scaled, verbose=0)

# Convert predictions and actual values back to original units
y_pred = scaler.inverse_transform(
    y_pred_scaled.reshape(-1, 1)
).ravel()

y_actual = scaler.inverse_transform(
    y_test_scaled.reshape(-1, 1)
).ravel()

# Calculate evaluation metrics
mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

# Avoid division by zero when calculating MAPE
nonzero = y_actual != 0
mape = np.mean(
    np.abs((y_actual[nonzero] - y_pred[nonzero]) / y_actual[nonzero])
) * 100

print("LSTM Model Evaluation")
print("---------------------")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

LSTM Model Evaluation
---------------------
MAE:  51.76
RMSE: 74.55
MAPE: 12.92%


In [23]:
import os

model_dir = "../models"

print("Checking saved LSTM model files:")

for filename in os.listdir(model_dir):
    if "lstm" in filename.lower() or "traffic" in filename.lower():
        print(filename)

Checking saved LSTM model files:
traffic_volume_features.json
traffic_volume_random_forest.pkl


In [24]:
import os

model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

lstm_path = os.path.join(
    model_dir, "traffic_volume_lstm.keras"
)

model.save(lstm_path)

print("LSTM model saved:", os.path.exists(lstm_path))
print("Saved path:", os.path.abspath(lstm_path))

LSTM model saved: True
Saved path: C:\Users\pathi\Desktop\FlowCast\models\traffic_volume_lstm.keras


In [25]:
import os

print("LSTM model exists:",
      os.path.exists("../models/traffic_volume_lstm.keras"))

print("Scaler exists:",
      os.path.exists("../models/traffic_volume_lstm_scaler.pkl"))

LSTM model exists: True
Scaler exists: False


In [26]:
import os
import joblib

model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

scaler_path = os.path.join(
    model_dir,
    "traffic_volume_lstm_scaler.pkl"
)

joblib.dump(scaler, scaler_path)

print("Scaler saved:", os.path.exists(scaler_path))
print("Saved path:", os.path.abspath(scaler_path))

Scaler saved: True
Saved path: C:\Users\pathi\Desktop\FlowCast\models\traffic_volume_lstm_scaler.pkl


In [27]:
import os

files = [
    "../models/traffic_volume_lstm.keras",
    "../models/traffic_volume_lstm_scaler.pkl",
    "../models/traffic_volume_random_forest.pkl",
    "../models/traffic_volume_features.json"
]

for file in files:
    print(os.path.basename(file), "->", os.path.exists(file))

traffic_volume_lstm.keras -> True
traffic_volume_lstm_scaler.pkl -> True
traffic_volume_random_forest.pkl -> True
traffic_volume_features.json -> True


In [1]:
from pathlib import Path

model_dir = Path("../models")

print("Models folder:", model_dir.resolve())
print("Folder exists:", model_dir.exists())

if model_dir.exists():
    for file in sorted(model_dir.iterdir()):
        if "traffic_volume_lstm" in file.name.lower():
            print(file.name, "|", file.stat().st_size, "bytes")

Models folder: C:\Users\pathi\Desktop\FlowCast\models
Folder exists: True
traffic_volume_lstm.keras | 87993 bytes
traffic_volume_lstm_scaler.pkl | 667 bytes


In [2]:
from pathlib import Path

project_dir = Path.home() / "Desktop" / "FlowCast"
models_dir = project_dir / "models"

print("Project folder exists:", project_dir.exists())
print("Models folder exists:", models_dir.exists())

print("\nSaved model files:")
if models_dir.exists():
    for file in sorted(models_dir.iterdir()):
        if file.is_file():
            print(f"{file.name} — {file.stat().st_size:,} bytes")
else:
    print("Models folder not found. Check the project path.")

Project folder exists: True
Models folder exists: True

Saved model files:
accident_risk_model.joblib — 64,018,986 bytes
congestion_classifier.pkl — 134,168,345 bytes
congestion_features.json — 191 bytes
traffic_volume_features.json — 338 bytes
traffic_volume_lstm.keras — 87,993 bytes
traffic_volume_lstm_scaler.pkl — 667 bytes
traffic_volume_random_forest.pkl — 558,672,657 bytes


In [1]:
import json
from pathlib import Path

features_path = Path("../models/traffic_volume_features.json")

print("Feature file exists:", features_path.exists())

if features_path.exists():
    with open(features_path, "r", encoding="utf-8") as f:
        features = json.load(f)
    print(json.dumps(features, indent=2))

Feature file exists: True
[
  "traffic_volume_lag_1",
  "traffic_volume_lag_2",
  "traffic_volume_lag_4",
  "avg_speed_lag_1",
  "avg_speed_lag_2",
  "traffic_volume_rolling_mean_4",
  "avg_speed_rolling_mean_4",
  "hour",
  "minute",
  "day_of_week",
  "month",
  "is_weekend",
  "is_morning_peak",
  "is_evening_peak"
]


In [3]:
from pathlib import Path
import joblib

project_dir = Path.cwd().parent
model_path = project_dir / "models" / "traffic_volume_random_forest.pkl"

print("Model exists:", model_path.exists())

if model_path.exists():
    rf_model = joblib.load(model_path)
    print("Model type:", type(rf_model).__name__)
    print("Expected feature count:", rf_model.n_features_in_)
    print("Expected feature names:",
          getattr(rf_model, "feature_names_in_", "Not stored in model"))

Model exists: True
Model type: RandomForestRegressor
Expected feature count: 14
Expected feature names: ['traffic_volume_lag_1' 'traffic_volume_lag_2' 'traffic_volume_lag_4'
 'avg_speed_lag_1' 'avg_speed_lag_2' 'traffic_volume_rolling_mean_4'
 'avg_speed_rolling_mean_4' 'hour' 'minute' 'day_of_week' 'month'
 'is_weekend' 'is_morning_peak' 'is_evening_peak']


In [4]:
print("Features in JSON:", len(features))
print("Features expected by model:", rf_model.n_features_in_)

print("\nJSON feature list:")
for i, feature in enumerate(features, start=1):
    print(i, feature)

print("\nModel feature list:")
print(list(rf_model.feature_names_in_))

Features in JSON: 14
Features expected by model: 14

JSON feature list:
1 traffic_volume_lag_1
2 traffic_volume_lag_2
3 traffic_volume_lag_4
4 avg_speed_lag_1
5 avg_speed_lag_2
6 traffic_volume_rolling_mean_4
7 avg_speed_rolling_mean_4
8 hour
9 minute
10 day_of_week
11 month
12 is_weekend
13 is_morning_peak
14 is_evening_peak

Model feature list:
['traffic_volume_lag_1', 'traffic_volume_lag_2', 'traffic_volume_lag_4', 'avg_speed_lag_1', 'avg_speed_lag_2', 'traffic_volume_rolling_mean_4', 'avg_speed_rolling_mean_4', 'hour', 'minute', 'day_of_week', 'month', 'is_weekend', 'is_morning_peak', 'is_evening_peak']


In [5]:
import pandas as pd

# Create a test input using the exact saved feature order
feature_names = list(rf_model.feature_names_in_)

# Use the latest available training/test data if it exists
print("Model expects", len(feature_names), "features:")
print(feature_names)

print("\nModel loaded successfully. Ready for prediction testing.")

Model expects 14 features:
['traffic_volume_lag_1', 'traffic_volume_lag_2', 'traffic_volume_lag_4', 'avg_speed_lag_1', 'avg_speed_lag_2', 'traffic_volume_rolling_mean_4', 'avg_speed_rolling_mean_4', 'hour', 'minute', 'day_of_week', 'month', 'is_weekend', 'is_morning_peak', 'is_evening_peak']

Model loaded successfully. Ready for prediction testing.


In [6]:
import json
from pathlib import Path

features_path = Path("../models/traffic_volume_features.json")

print("File exists:", features_path.exists())

if features_path.exists():
    with open(features_path, "r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2))

File exists: True
[
  "traffic_volume_lag_1",
  "traffic_volume_lag_2",
  "traffic_volume_lag_4",
  "avg_speed_lag_1",
  "avg_speed_lag_2",
  "traffic_volume_rolling_mean_4",
  "avg_speed_rolling_mean_4",
  "hour",
  "minute",
  "day_of_week",
  "month",
  "is_weekend",
  "is_morning_peak",
  "is_evening_peak"
]


In [3]:
import pandas as pd

df = pd.read_csv("../data/traffic_sensor_log.csv")
df.columns = df.columns.str.strip()

df["timestamp"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["time"].astype(str),
    errors="coerce"
)

for road, group in df.groupby("road_id"):
    intervals = (
        group.sort_values("timestamp")["timestamp"]
        .diff()
        .dropna()
    )
    print(f"\nRoad: {road}")
    print(intervals.value_counts().head(5))


Road: NL-001
timestamp
0 days 00:30:00    6874
0 days 01:00:00     179
0 days 00:00:00      53
0 days 01:30:00       5
Name: count, dtype: int64

Road: NL-002
timestamp
0 days 00:30:00    6917
0 days 01:00:00     159
0 days 00:00:00      74
0 days 01:30:00       4
Name: count, dtype: int64

Road: NL-003
timestamp
0 days 00:30:00    6881
0 days 01:00:00     180
0 days 00:00:00      66
0 days 01:30:00       2
Name: count, dtype: int64

Road: NL-004
timestamp
0 days 00:30:00    6936
0 days 01:00:00     151
0 days 00:00:00      78
0 days 01:30:00       3
Name: count, dtype: int64

Road: NL-005
timestamp
0 days 00:30:00    6882
0 days 01:00:00     178
0 days 00:00:00      72
0 days 01:30:00       3
Name: count, dtype: int64

Road: NL-006
timestamp
0 days 00:30:00    6862
0 days 01:00:00     185
0 days 00:00:00      71
0 days 01:30:00       5
Name: count, dtype: int64

Road: NL-007
timestamp
0 days 00:30:00    6899
0 days 01:00:00     168
0 days 00:00:00      71
0 days 01:30:00       4
Name

In [2]:
print("Available columns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Available columns:
['road_id', 'road_name', 'latitude', 'longitude', 'weather_station_id', 'date', 'time', 'traffic_volume', 'vehicle_count', 'vehicle_type_dist', 'avg_speed', 'occupancy', 'congestion_level', 'travel_time', 'accident_count', 'signal_timing', 'road_capacity']

First 5 rows:


,road_id,road_name,latitude,longitude,weather_station_id,date,time,traffic_volume,vehicle_count,vehicle_type_dist,avg_speed,occupancy,congestion_level,travel_time,accident_count,signal_timing,road_capacity
0,NL-007,Kingsway @ Park,28.647903,77.149142,WS-NORTH,2025-02-26,07:30,706.0,706,"{""2W"": 0.37, ""Car"": 0.43, ""LCV"": 0.11, ""HCV"": ...",31.2,59.6,Moderate,2.12,0,54,1928
1,NL-014,Industrial Link B,28.594749,77.253501,WS-CENTRAL,2025-03-23,21:00,308.0,308,"{""2W"": 0.43, ""Car"": 0.39, ""LCV"": 0.08, ""HCV"": ...",53.5,16.8,Free-flow,2.03,0,32,2327
2,NL-022,Old Town @ Clock,28.674521,77.219835,WS-NORTH,2025-04-05,11:00,342.0,342,"{""2W"": 0.4, ""Car"": 0.33, ""LCV"": 0.15, ""HCV"": 0...",49.9,27.5,Free-flow,3.46,0,38,1771
3,NL-006,Central Flyover E,28.622342,77.258257,WS-SOUTH,2025-05-04,01:30,187.0,187,"{""2W"": 0.41, ""Car"": 0.39, ""LCV"": 0.11, ""HCV"": ...",55.1,10.7,Free-flow,2.18,0,30,2285
4,NL-014,Industrial Link B,28.594749,77.253501,WS-CENTRAL,2025-01-08,12:30,490.0,490,"{""2W"": 0.39, ""Car"": 0.4, ""LCV"": 0.1, ""HCV"": 0.11}",45.9,27.7,Free-flow,2.37,0,45,2327
